# 12. `not responding` (node, status)

In [2]:
from common import *

shape, log = show(12)

log shape 85: node | status | not responding
10,628 logs (2.6%) from 257 nodes, 2003-12-26 21:07:30 to 2006-04-26 22:17:28
log 106433 at 2004-07-15 12:17:53: 400027 node-221 node status 1089893873 1 not responding


What the node logged in the day before

In [3]:
print("Scary. But maybe it just indicates a reboot?")
display(logs.node_timeline(log.node, log.ts - timedelta(days=1), log.ts))

Scary. But maybe it just indicates a reboot?


[TimelineEntry(component='node', state='status', template='running', log_shape_id=84, first_ts='2004-07-14 13:30:25', last_ts='2004-07-14 13:30:25', count=1),
 TimelineEntry(component='node', state='status', template='not responding', log_shape_id=85, first_ts='2004-07-15 12:17:53', last_ts='2004-07-15 12:17:53', count=1)]

Commands anywhere in the cluster in the day before

In [4]:
print("maybe the reboot command is logged with a different component. serach for text: 'boot' or 'reboot' or 'shutdown' or 'poweroff' or 'halt' or 'init'")
results = []
for text in ["boot", "reboot", "shutdown", "poweroff", "halt", "init"]:
    results.extend(logs.search(log.ts - timedelta(days=1), log.ts, text=text))
print(results)
assert results == []
print("Looks like no reboot. Sound the alarms.")

maybe the reboot command is logged with a different component. serach for text: 'boot' or 'reboot' or 'shutdown' or 'poweroff' or 'halt' or 'init'
[]
Looks like no reboot. Sound the alarms.


OTher nodes that stop responding in the 10 minutes before our log

In [6]:
down = logs.stats(log.ts - 10 * minute, log.ts, log_shape_id=shape.log_shape_id)
print(down)

LogStats(n_logs=203, n_nodes=203, first_seen='2004-07-15 12:15:16', last_seen='2004-07-15 12:17:53')


Everything the cluster logged in that window

In [8]:
display(shapes_between(log.ts - hour, log.ts))

,component,state,template,n_logs,n_sources,first,last
0,gige,temperature,normal,6,4,2004-07-15 11:27:05,2004-07-15 11:59:34
1,gige,temperature,warning,6,4,2004-07-15 11:28:22,2004-07-15 12:04:34
2,switch_module,error,Linkerror event interval expired,531,8,2004-07-15 12:14:21,2004-07-15 12:17:53
3,partition,status,blocked,1,1,2004-07-15 12:15:00,2004-07-15 12:15:00
4,node,status,not responding,203,203,2004-07-15 12:15:16,2004-07-15 12:17:53
5,clusterfilesystem,clusterfilesystem.no_server,ClusterFileSystem: There is no server for ServerFileSystem domain storage<N>,56,1,2004-07-15 12:15:22,2004-07-15 12:15:32
6,domain,status,inconsistent nodesets <NODE> <HEX> <ok>,5,5,2004-07-15 12:15:30,2004-07-15 12:17:30
7,node,status,configured out,174,174,2004-07-15 12:15:31,2004-07-15 12:17:31


gige temperature and Linkerror are usually noise. We can see that there was this `partition status blocked` log, and then a cascade off errors including `not reponding` on 203 nodes and the `ClusterFileSystem: There is no server for ServerFileSystem domain storage<N>` nessage appearing on the `clusterfilesystem` node a lot of times.kater the nodes are configured out.

# Verdict: Critical. An outage of most of the nodes if not all. However we should notice that it is one alarm for the whole cluster, not 256 separate alarms because this isnt just a node-221 problem.
The cause of this outage is probably the `full partition status blocked` log.